# 🎯 Fitts' Law Analysis - BY VARIANCE TYPE

## 📊 What This Notebook Does:
- Analyzes data from ALL 12 participants
- **Separates results by variance level (Low ~1.0, Medium ~7.0, High ~12.5)**
- **Finds MAXIMUM throughput across variance levels**
- **Generates individual participant tables**

## 🎓 For Professor's Questions:
1. ✅ Shows throughput separately for each variance type
2. ✅ Identifies which variance gives best performance
3. ✅ Provides per-participant data table

## 🚀 How to Use:
1. Run Cell 1 (Setup)
2. Run Cell 2 (Upload) - Upload fitts-student-data.zip
3. Run remaining cells to see results!

In [ ]:
# 1️⃣ Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os
import glob
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ Packages loaded!")

In [ ]:
# 2️⃣ Upload Data
from google.colab import files
import zipfile

print("📁 Upload your fitts-student-data.zip file:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"\n📦 Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✅ Extracted!")

In [ ]:
# 3️⃣ Load ALL Data from ALL 12 Participants
print("🔍 Loading data from ALL 12 participants...")
print("="*80)

student_folders = [d for d in os.listdir('fitts-student-data')
                   if os.path.isdir(os.path.join('fitts-student-data', d))]

print(f"\nFound {len(student_folders)} student folders\n")

# Load raw data
raw_data_by_student = {}
for student in student_folders:
    student_path = os.path.join('fitts-student-data', student)
    raw_files = glob.glob(os.path.join(student_path, '*raw-data*.csv'))
    
    if raw_files:
        df = pd.read_csv(raw_files[0])
        df['ParticipantID'] = student
        raw_data_by_student[student] = df
        print(f"   ✅ {student}: {len(df)} trials (raw data)")

# Load pre-calculated results
results_by_student = {}
for student in student_folders:
    student_path = os.path.join('fitts-student-data', student)
    results_files = glob.glob(os.path.join(student_path, '*results*.csv'))
    
    if results_files:
        df = pd.read_csv(results_files[0])
        df['ParticipantID'] = student
        results_by_student[student] = df
        print(f"   ✅ {student}: {len(df)} conditions (pre-calculated results)")

print(f"\n📊 Summary:")
print(f"   Students with raw data: {len(raw_data_by_student)}")
print(f"   Students with results: {len(results_by_student)}")

In [ ]:
# 4️⃣ Calculate Results for ALL 12 Participants
print("\n" + "="*80)
print("📊 PROCESSING ALL 12 PARTICIPANTS")
print("="*80)

all_results = []

# For each student
for student in student_folders:
    print(f"\n🔄 Processing {student}...")
    
    # Check if we have pre-calculated results
    if student in results_by_student:
        print(f"   ✅ Using pre-calculated results ({len(results_by_student[student])} conditions)")
        all_results.append(results_by_student[student])
    
    # Otherwise, calculate from raw data
    elif student in raw_data_by_student:
        print(f"   🔧 Calculating from raw data ({len(raw_data_by_student[student])} trials)...")
        
        raw_df = raw_data_by_student[student]
        student_results = []
        
        # Group by filter and layout conditions
        grouping_cols = ['FilterType', 'TargetSize', 'Amplitude']
        if 'PairVariance' in raw_df.columns:
            grouping_cols.insert(0, 'PairVariance')
        grouping_cols = [col for col in grouping_cols if col in raw_df.columns]
        
        for group_keys, group in raw_df.groupby(grouping_cols):
            # Calculate Fitts metrics
            Ae = group['ActualAmplitude'].mean()
            
            # ISO 9241-9 directional projection: project endpoint deviation onto movement direction
            if 'Direction' in group.columns and 'TargetX' in group.columns:
                theta_rad = np.radians(group['Direction'].astype(float))
                dx = group['SelectionX'] - group['TargetX']
                dy = group['SelectionY'] - group['TargetY']
                projections = dx * np.cos(theta_rad) + dy * np.sin(theta_rad)
                We = 4.133 * projections.std()
            else:
                dx = group['SelectionX'] - group['TargetX']
                dy = group['SelectionY'] - group['TargetY']
                distances = np.sqrt(dx**2 + dy**2)
                We = 4.133 * distances.std()
            
            IDe = np.log2(Ae / We + 1) if We > 0 else 0
            MeanMT = group['MovementTime'].mean()
            TP = IDe / MeanMT if MeanMT > 0 else 0
            
            result = {
                'ParticipantID': student,
                'PairVariance': group_keys[0] if 'PairVariance' in grouping_cols else None,
                'FilterType': group_keys[1] if 'PairVariance' in grouping_cols else group_keys[0],
                'TargetSize': group_keys[2] if 'PairVariance' in grouping_cols else group_keys[1],
                'Amplitude': group_keys[3] if 'PairVariance' in grouping_cols else group_keys[2],
                'N': len(group),
                'MeanMT': MeanMT,
                'Ae': Ae,
                'We': We,
                'IDe': IDe,
                'TP': TP
            }
            student_results.append(result)
        
        student_df = pd.DataFrame(student_results)
        all_results.append(student_df)
        print(f"   ✅ Calculated {len(student_df)} conditions")
    
    else:
        print(f"   ⚠️  No data found for {student}")

# Combine all results
results_data = pd.concat(all_results, ignore_index=True)

print(f"\n" + "="*80)
print(f"✅ COMPLETE! Analyzed {results_data['ParticipantID'].nunique()} participants")
print(f"   Total conditions: {len(results_data)}")
print(f"   Participants: {sorted(results_data['ParticipantID'].unique())}")
print("="*80)

# Show preview
print("\n📊 Data preview:")
display(results_data.head(10))

In [ ]:
# 5️⃣ 🎯 ANALYSIS BY VARIANCE TYPE - PROFESSOR'S QUESTION #1
print("="*80)
print("🎯 THROUGHPUT BY VARIANCE TYPE (Professor's Question #1)")
print("="*80)

if 'PairVariance' in results_data.columns and 'FilterType' in results_data.columns:
    # Group by variance and filter
    variance_analysis = results_data.groupby(['PairVariance', 'FilterType']).agg({
        'TP': ['mean', 'std', 'count'],
        'MeanMT': ['mean', 'std']
    }).round(4)
    
    print("\n📊 Throughput and Movement Time by Variance Level:\n")
    display(variance_analysis)
    
    # Find best variance for each filter
    print("\n" + "="*80)
    print("🏆 MAXIMUM THROUGHPUT BY VARIANCE (Answer to Professor)")
    print("="*80)
    
    variance_summary = results_data.groupby(['PairVariance', 'FilterType'])['TP'].mean().reset_index()
    
    for filter_type in variance_summary['FilterType'].unique():
        filter_data = variance_summary[variance_summary['FilterType'] == filter_type]
        best_row = filter_data.loc[filter_data['TP'].idxmax()]
        
        print(f"\n{filter_type.upper()} Filter:")
        print(f"   Best Variance: {best_row['PairVariance']}")
        print(f"   Max Throughput: {best_row['TP']:.4f} bits/s")
        
        # Show all variance levels for comparison
        print(f"\n   All variance levels:")
        for _, row in filter_data.iterrows():
            marker = "🏆" if row['TP'] == best_row['TP'] else "  "
            print(f"   {marker} Variance {row['PairVariance']}: {row['TP']:.4f} bits/s")
    
    # Overall winner
    print("\n" + "="*80)
    best_overall = variance_summary.loc[variance_summary['TP'].idxmax()]
    print(f"🎖️  OVERALL BEST CONFIGURATION:")
    print(f"   Filter: {best_overall['FilterType']}")
    print(f"   Variance: {best_overall['PairVariance']}")
    print(f"   Throughput: {best_overall['TP']:.4f} bits/s")
    print("="*80)
    
else:
    print("⚠️  No variance information found in data")
    print("   Showing overall results instead...\n")
    
    if 'FilterType' in results_data.columns:
        overall = results_data.groupby('FilterType').agg({
            'TP': ['mean', 'std'],
            'MeanMT': ['mean', 'std']
        }).round(4)
        display(overall)

In [ ]:
# 6️⃣ 👥 INDIVIDUAL PARTICIPANT DATA - PROFESSOR'S QUESTION #2
print("="*80)
print("👥 INDIVIDUAL PARTICIPANT DATA (Professor's Question #2)")
print("="*80)

if 'FilterType' in results_data.columns and 'PairVariance' in results_data.columns:
    # Get BOTH average and best values for each participant, filter, and variance combination
    participant_data_avg = results_data.groupby(['ParticipantID', 'FilterType', 'PairVariance']).agg({
        'TP': 'mean',      # Average throughput
        'MeanMT': 'mean'   # Average movement time
    }).reset_index()
    
    participant_data_best = results_data.groupby(['ParticipantID', 'FilterType', 'PairVariance']).agg({
        'TP': 'max',       # Maximum (best) throughput - HIGHER is better
        'MeanMT': 'min'    # Minimum (best) movement time - LOWER is better
    }).reset_index()
    
    # Create column names for each filter + variance combination
    # Format: exponential_var1.0_TP_avg, exponential_var1.0_TP_max, exponential_var1.0_MT_avg, exponential_var1.0_MT_max
    participant_table = []
    
    for participant in participant_data_avg['ParticipantID'].unique():
        row = {'ParticipantID': participant}
        
        participant_subset_avg = participant_data_avg[participant_data_avg['ParticipantID'] == participant]
        participant_subset_best = participant_data_best[participant_data_best['ParticipantID'] == participant]
        
        for _, data_row_avg in participant_subset_avg.iterrows():
            filter_type = data_row_avg['FilterType']
            variance = data_row_avg['PairVariance']
            tp_avg = data_row_avg['TP']
            mt_avg = data_row_avg['MeanMT']
            
            # Find corresponding best values (max TP, min MT)
            data_row_best = participant_subset_best[
                (participant_subset_best['FilterType'] == filter_type) & 
                (participant_subset_best['PairVariance'] == variance)
            ]
            
            if len(data_row_best) > 0:
                tp_best = data_row_best.iloc[0]['TP']
                mt_best = data_row_best.iloc[0]['MeanMT']
            else:
                tp_best = tp_avg
                mt_best = mt_avg
            
            # Create column names with both avg and best
            row[f'{filter_type}_var{variance}_TP_avg'] = round(tp_avg, 4)
            row[f'{filter_type}_var{variance}_TP_best'] = round(tp_best, 4)
            row[f'{filter_type}_var{variance}_MT_avg'] = round(mt_avg, 4)
            row[f'{filter_type}_var{variance}_MT_best'] = round(mt_best, 4)
        
        participant_table.append(row)
    
    participant_table = pd.DataFrame(participant_table)
    
    # Sort columns for better readability
    # Order: ParticipantID, then exponential (var 1, 7, 12.5), then oneEuro (var 1, 7, 12.5)
    # For each variance: TP_avg, TP_best, MT_avg, MT_best
    col_order = ['ParticipantID']
    for filter_type in sorted(participant_data_avg['FilterType'].unique()):
        for variance in sorted(participant_data_avg['PairVariance'].unique()):
            col_order.append(f'{filter_type}_var{variance}_TP_avg')
            col_order.append(f'{filter_type}_var{variance}_TP_best')
            col_order.append(f'{filter_type}_var{variance}_MT_avg')
            col_order.append(f'{filter_type}_var{variance}_MT_best')
    
    # Only include columns that exist
    col_order = [col for col in col_order if col in participant_table.columns]
    participant_table = participant_table[col_order]
    
    print("\n📋 Throughput and Movement Time per Participant (All Variance Levels):")
    print("Format: filterName_varX.X_TP_avg (average), filterName_varX.X_TP_best (max - higher is better)")
    print("        filterName_varX.X_MT_avg (average), filterName_varX.X_MT_best (min - lower is better)")
    print(f"Total: {len(participant_table)} participants × 2 filters × 3 variances × 4 metrics = {len(participant_table) * 24} values\n")
    display(participant_table)
    
    # Save to CSV
    participant_table.to_csv('participant_data_for_professor.csv', index=False)
    print("\n💾 Saved: participant_data_for_professor.csv")
    print("   Columns: ParticipantID + (TP_avg, TP_best [max], MT_avg, MT_best [min] for each filter × variance)")
    
    # Summary statistics by variance level
    print("\n" + "="*80)
    print("📊 SUMMARY STATISTICS BY VARIANCE:")
    print("="*80)
    
    for variance in sorted(participant_data_avg['PairVariance'].unique()):
        print(f"\n📊 Variance {variance}:")
        variance_data_avg = participant_data_avg[participant_data_avg['PairVariance'] == variance]
        variance_data_best = participant_data_best[participant_data_best['PairVariance'] == variance]
        
        for filter_type in sorted(variance_data_avg['FilterType'].unique()):
            filter_var_data_avg = variance_data_avg[variance_data_avg['FilterType'] == filter_type]
            filter_var_data_best = variance_data_best[variance_data_best['FilterType'] == filter_type]
            print(f"   {filter_type}:")
            print(f"      TP (avg): {filter_var_data_avg['TP'].mean():.4f} ± {filter_var_data_avg['TP'].std():.4f} bits/s")
            print(f"      TP (best): {filter_var_data_best['TP'].mean():.4f} ± {filter_var_data_best['TP'].std():.4f} bits/s [max]")
            print(f"      MT (avg): {filter_var_data_avg['MeanMT'].mean():.4f} ± {filter_var_data_avg['MeanMT'].std():.4f} s")
            print(f"      MT (best): {filter_var_data_best['MeanMT'].mean():.4f} ± {filter_var_data_best['MeanMT'].std():.4f} s [min]")
    
else:
    print("⚠️  No filter type or variance information found in data")

In [ ]:
# 7️⃣ Statistical Tests by Variance
print("="*80)
print("📊 STATISTICAL TESTS BY VARIANCE LEVEL")
print("="*80)

if 'PairVariance' in results_data.columns and 'FilterType' in results_data.columns:
    variance_levels = sorted(results_data['PairVariance'].unique())
    filters = sorted(results_data['FilterType'].unique())
    
    if len(filters) == 2:
        f1, f2 = filters
        
        for var_level in variance_levels:
            print(f"\n{'='*80}")
            print(f"📊 Variance Level: {var_level}")
            print(f"{'='*80}")
            
            var_data = results_data[results_data['PairVariance'] == var_level]
            
            # Get throughput by participant for this variance level
            tp1 = var_data[var_data['FilterType'] == f1].groupby('ParticipantID')['TP'].mean()
            tp2 = var_data[var_data['FilterType'] == f2].groupby('ParticipantID')['TP'].mean()
            
            print(f"\n{f1}: {tp1.mean():.4f} ± {tp1.std():.4f} bits/s (N={len(tp1)} participants)")
            print(f"{f2}: {tp2.mean():.4f} ± {tp2.std():.4f} bits/s (N={len(tp2)} participants)")
            
            # Find common participants
            common = set(tp1.index) & set(tp2.index)
            
            if len(common) > 1:
                tp1_vals = [tp1[p] for p in common]
                tp2_vals = [tp2[p] for p in common]
                
                # Paired t-test
                t_stat, p_val = stats.ttest_rel(tp1_vals, tp2_vals)
                
                print(f"\nPaired t-test (N={len(common)}): t({len(common)-1})={t_stat:.3f}, p={p_val:.4f}")
                
                if p_val < 0.05:
                    better = f1 if np.mean(tp1_vals) > np.mean(tp2_vals) else f2
                    worse = f2 if better == f1 else f1
                    improvement = abs((np.mean(tp1_vals) - np.mean(tp2_vals)) / min(np.mean(tp1_vals), np.mean(tp2_vals)) * 100)
                    print(f"✅ SIGNIFICANT (p < 0.05)")
                    print(f"🏆 {better} is {improvement:.1f}% better")
                else:
                    print(f"❌ Not significant (p >= 0.05)")
                
                # Cohen's d
                diff = np.array(tp1_vals) - np.array(tp2_vals)
                cohens_d = np.mean(diff) / np.std(diff, ddof=1)
                print(f"Effect size (Cohen's d): {abs(cohens_d):.3f}")
                
                if abs(cohens_d) < 0.2:
                    print(f"   → Small effect")
                elif abs(cohens_d) < 0.5:
                    print(f"   → Medium effect")
                else:
                    print(f"   → Large effect")

In [ ]:
# 8️⃣ Graph: Throughput by Variance and Filter
if 'PairVariance' in results_data.columns and 'FilterType' in results_data.columns:
    plt.figure(figsize=(12, 6))
    
    # Create grouped bar plot
    variance_summary = results_data.groupby(['PairVariance', 'FilterType'])['TP'].mean().reset_index()
    
    sns.barplot(data=variance_summary, x='PairVariance', y='TP', hue='FilterType', palette='Set2')
    
    plt.title('Throughput by Variance Level and Filter Type (N=12)', fontsize=16, fontweight='bold')
    plt.xlabel('Variance Level', fontsize=12)
    plt.ylabel('Throughput (bits/s)', fontsize=12)
    plt.legend(title='Filter Type')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('throughput_by_variance_N12.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("💾 Saved: throughput_by_variance_N12.png")

In [ ]:
# 9️⃣ Graph: Movement Time by Variance and Filter
if 'PairVariance' in results_data.columns and 'FilterType' in results_data.columns:
    plt.figure(figsize=(12, 6))
    
    # Create grouped bar plot
    mt_summary = results_data.groupby(['PairVariance', 'FilterType'])['MeanMT'].mean().reset_index()
    
    sns.barplot(data=mt_summary, x='PairVariance', y='MeanMT', hue='FilterType', palette='Set2')
    
    plt.title('Movement Time by Variance Level and Filter Type (N=12)', fontsize=16, fontweight='bold')
    plt.xlabel('Variance Level', fontsize=12)
    plt.ylabel('Movement Time (s)', fontsize=12)
    plt.legend(title='Filter Type')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('movement_time_by_variance_N12.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("💾 Saved: movement_time_by_variance_N12.png")

In [ ]:
# 🔟 Graph: Individual Participant Performance
if 'FilterType' in results_data.columns:
    # Get best throughput for each participant and filter
    participant_best = results_data.groupby(['ParticipantID', 'FilterType'])['TP'].max().reset_index()
    
    plt.figure(figsize=(16, 6))
    sns.barplot(data=participant_best, x='ParticipantID', y='TP', hue='FilterType', palette='Set2')
    plt.title('Best Throughput per Participant (N=12)', fontsize=16, fontweight='bold')
    plt.ylabel('Throughput (bits/s)', fontsize=12)
    plt.xlabel('Participant', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Filter')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('participant_best_throughput_N12.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("💾 Saved: participant_best_throughput_N12.png")

In [ ]:
# 1️⃣1️⃣ Export All Results
print("="*80)
print("💾 EXPORTING RESULTS")
print("="*80)

# 1. Summary by variance and filter
if 'PairVariance' in results_data.columns and 'FilterType' in results_data.columns:
    variance_export = results_data.groupby(['PairVariance', 'FilterType']).agg({
        'TP': ['mean', 'std', 'min', 'max', 'count'],
        'MeanMT': ['mean', 'std', 'min', 'max']
    }).round(4)
    variance_export.to_csv('summary_by_variance_N12.csv')
    print("✅ Saved: summary_by_variance_N12.csv")

# 2. Individual participant data (already saved above)
print("✅ Saved: participant_data_for_professor.csv")

# 3. All raw results
results_data.to_csv('all_results_by_variance_N12.csv', index=False)
print("✅ Saved: all_results_by_variance_N12.csv")

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE!")
print("="*80)
print("\n📊 Files Generated:")
print("   • 3 graphs (PNG, 300 DPI)")
print("   • 3 CSV tables")
print("   • Statistical test results above")
print("\n🎓 Ready to send to professor!")
print("   → Check 'participant_data_for_professor.csv' for Question #2")
print("   → Check output above for Question #1 (max throughput by variance)")

In [ ]:
# 1️⃣2️⃣ 📧 EMAIL SUMMARY FOR PROFESSOR
print("="*80)
print("📧 EMAIL SUMMARY - COPY THIS TO SEND TO PROFESSOR")
print("="*80)

if 'PairVariance' in results_data.columns and 'FilterType' in results_data.columns:
    # Find best configuration
    variance_summary = results_data.groupby(['PairVariance', 'FilterType'])['TP'].mean().reset_index()
    best_overall = variance_summary.loc[variance_summary['TP'].idxmax()]
    
    print("\nDear Professor,\n")
    print("Here are the answers to your questions:\n")
    print("1. THROUGHPUT BY VARIANCE TYPE:\n")
    
    for filter_type in sorted(variance_summary['FilterType'].unique()):
        filter_data = variance_summary[variance_summary['FilterType'] == filter_type]
        best_row = filter_data.loc[filter_data['TP'].idxmax()]
        
        print(f"   {filter_type.upper()} Filter:")
        for _, row in filter_data.iterrows():
            marker = "(BEST)" if row['TP'] == best_row['TP'] else ""
            print(f"      Variance {row['PairVariance']}: {row['TP']:.4f} bits/s {marker}")
        print()
    
    print(f"   MAXIMUM THROUGHPUT: {best_overall['TP']:.4f} bits/s")
    print(f"   Best Configuration: {best_overall['FilterType']} filter with variance {best_overall['PairVariance']}\n")
    
    print("2. INDIVIDUAL PARTICIPANT DATA:")
    print("   Please see attached 'participant_data_for_professor.csv'\n")
    
    print("Graphs attached:")
    print("   • throughput_by_variance_N12.png")
    print("   • movement_time_by_variance_N12.png")
    print("   • participant_best_throughput_N12.png\n")
    
    print("Best regards,")
    print("Soha")
    print("\n" + "="*80)